In [2]:
pip install ucimlrepo

In [4]:
import io
import zipfile
import pandas as pd
import requests

# Direct URL to the dataset zip on the UCI Machine Learning Repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00359/NewsAggregatorDataset.zip"

print("Downloading dataset...")
response = requests.get(url)
response.raise_for_status()

# Extract and read newsCorpora.csv into a pandas DataFrame
print("Extracting and loading into pandas...")
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # The file is tab-separated and does not have a header row in the raw text
    with z.open("newsCorpora.csv") as f:
        df = pd.read_csv(
            f,
            sep="\t",
            header=None,
            names=[
                "ID",
                "TITLE",
                "URL",
                "PUBLISHER",
                "CATEGORY",
                "STORY",
                "HOSTNAME",
                "TIMESTAMP",
            ],
        )

# Display the first few rows
print(df.head())
print(f"Total records loaded: {len(df):,}")

Extracting and loading into pandas...
   ID                                              TITLE  \
0   1  Fed official says weak data caused by weather,...   
1   2  Fed's Charles Plosser sees high bar for change...   
2   3  US open: Stocks fall after Fed official hints ...   
3   4  Fed risks falling 'behind the curve', Charles ...   
4   5  Fed's Plosser: Nasty Weather Has Curbed Job Gr...   

                                                 URL          PUBLISHER  \
0  http://www.latimes.com/business/money/la-fi-mo...  Los Angeles Times   
1  http://www.livemint.com/Politics/H2EvwJSK2VE6O...           Livemint   
2  http://www.ifamagazine.com/news/us-open-stocks...       IFA Magazine   
3  http://www.ifamagazine.com/news/fed-risks-fall...       IFA Magazine   
4  http://www.moneynews.com/Economy/federal-reser...          Moneynews   

  CATEGORY                          STORY             HOSTNAME      TIMESTAMP  
0        b  ddUyU0VZz0BRneMioxUPQVP6sIxvM      www.latimes.com  139447

In [5]:
import pandas as pd

# 1. Convert timestamp from epoch milliseconds to datetime
df["DATETIME"] = pd.to_datetime(df["TIMESTAMP"], unit="ms")
df["DATE"] = df["DATETIME"].dt.date

# 2. Map category codes to readable names
category_map = {
    "b": "Business",
    "t": "Science & Tech",
    "e": "Entertainment",
    "m": "Health",
}
df["CATEGORY_NAME"] = df["CATEGORY"].map(category_map)

print(df[["DATETIME", "CATEGORY_NAME", "PUBLISHER", "TITLE"]].head(3))

                 DATETIME CATEGORY_NAME          PUBLISHER  \
0 2014-03-10 16:52:50.698      Business  Los Angeles Times   
1 2014-03-10 16:52:51.207      Business           Livemint   
2 2014-03-10 16:52:51.550      Business       IFA Magazine   

                                               TITLE  
0  Fed official says weak data caused by weather,...  
1  Fed's Charles Plosser sees high bar for change...  
2  US open: Stocks fall after Fed official hints ...  


In [6]:
# Group by STORY cluster to create macro-chunks of news coverage
story_chunks = (
    df.groupby("STORY")
    .agg(
        # Combine all headlines for this story into a single text block
        COMBINED_TITLES=(
            "TITLE",
            lambda x: " | ".join(x.astype(str).unique()),
        ),
        # Count how many publishers covered this story (proxy for importance/impact)
        COVERAGE_COUNT=("PUBLISHER", "count"),
        # List the unique publishers covering it
        PUBLISHERS=("PUBLISHER", lambda x: list(x.unique())),
        # Get the primary category and earliest timestamp
        CATEGORY=("CATEGORY_NAME", "first"),
        STORY_DATE=("DATE", "min"),
    )
    .reset_index()
)

print(story_chunks.head())

                           STORY  \
0  d--2ViHBj27LFNMHCePQDgJyCBjzM   
1  d--KGWIac5fEeVM67-ZRRhppFjOSM   
2  d--MozT4MsoFfIMgKu5_N58OF_f9M   
3  d--XtfmzZ2YqjSMvlrF2EEknl2keM   
4  d--wbowLE_mEcaMG2HSlJBQmHDBVM   

                                     COMBINED_TITLES  COVERAGE_COUNT  \
0  Devon Energy sells gas properties to Linn | De...              44   
1  Google launches price war in the cloud | 3 Pic...              26   
2  Selena Gomez's reunion with Bieber matter of c...             241   
3  Envoy: Ukraine getting closer to deal with IMF...              67   
4  China's 'shadow banking' sector valued at US$4...              91   

                                          PUBLISHERS        CATEGORY  \
0  [MSN NZ News, Chicago Sun-Times, Resource Inve...        Business   
1  [MarketWatch, ReadWrite, Gizbot, 24\/7 Wall St...  Science & Tech   
2  [ANINEWS, International Business Times AU, Won...   Entertainment   
3  [Thegardenisland.com, Fremont Tribune, New Str...        Bu

In [7]:
# Export aggregated story chunks to CSV for dashboard import
story_chunks.to_csv("news_bi_ready_data.csv", index=False)
print("Saved enriched data to news_bi_ready_data.csv!")

Saved enriched data to news_bi_ready_data.csv!
